# Phase 4 — MuJoCo 3D Visualization

**Project**: Quantum-Inspired Tensor-Network Compression of OpenVLA-7B  
**Challenge**: Global Quantum + AI Challenge 2026 — VW Group Enterprise Track  
**Phase**: 4 of 7 | **Execution**: Kaggle (GPU kernel, P100)

---

## What this notebook does

Drives a **dm_control humanoid** figure with action outputs from the chi=64
compressed OpenVLA-7B model, then renders and encodes side-by-side MP4 comparisons.

**Pipeline**:
```
lerobot/toto sample (image + language)
    → Compressed OpenVLA-7B (chi=64, from Phase 2 kernel_sources)
    → 7-DoF delta action [dx, dy, dz, droll, dpitch, dyaw, gripper]
    → action_to_joint_command()  (mujoco_bridge.py)
    → 21-DoF humanoid ctrl  (right arm driven; lower body PD-stabilised)
    → dm_control humanoid physics (CPU)
    → rendered frames → MP4 + GIF
```

**Joint mapping**: right arm actuators at indices 15 (shoulder1), 16 (shoulder2), 17 (elbow).
Lower body (indices 0–14) is stabilised by a proportional controller.

## Prerequisites

1. **GPU runtime** enabled.
2. **Kaggle Secrets**: `HF_TOKEN` and `GH_TOKEN`.
3. **Phase 2 kernel** (`benjaminbrumm/vw-phase2-compression`) must have a
   successful completed run (kernel_sources mounts its output at
   `/kaggle/input/vw-phase2-compression/`).

---

> **License**: OpenVLA-7B weights are subject to the LLaMA-2 Community License
> (non-commercial research use only).

In [ ]:
# ── Cell 1: Install / pin dependencies ───────────────────────────────────────
# Mirrors Phase 1-3: SM detection, torch==2.2.2+cu118 for P100 (sm_60),
# NCCL stub for libtorch_cuda.so undefined symbols, dm_control + EGL setup.
import subprocess, sys, os, ctypes, re, glob as _glob

def _pip(*args, check=True):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=check)

# ── 1. Detect GPU SM version BEFORE any torch import ─────────────────────────
_sm_ver = 70
try:
    _smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15,
    )
    if _smi.returncode == 0:
        _cap = _smi.stdout.strip().split('\n')[0].strip()
        _sm_ver = int(_cap.replace('.', ''))
        print(f'nvidia-smi: compute_cap={_cap}  (sm_{_sm_ver})')
except Exception as _e:
    print(f'nvidia-smi check skipped ({_e}) — assuming sm_70+')

# ── 2. Install ALL non-torch deps FIRST ──────────────────────────────────────
print('Installing non-torch deps ...')
_pip(
    'transformers==4.40.1',
    'tokenizers==0.19.1',
    'timm==0.9.10',
    'sentencepiece==0.1.99',
)
_pip(
    'accelerate>=0.27.0',
    'PyYAML',
    'tqdm',
    'av',
    'datasets',
    'matplotlib',
)
_pip(
    'dm-control>=1.0.20',
    'mujoco>=3.1.0',
    'imageio[ffmpeg]',
    'Pillow',
)
print('Non-torch deps installed.')

# ── 3. Pin SM-compatible torch LAST ──────────────────────────────────────────
if _sm_ver < 70:
    print(f'sm_{_sm_ver} < 70 → pinning torch==2.2.2+cu118 + torchvision==0.17.2+cu118 ...')
    _pip(
        'torch==2.2.2+cu118',
        'torchvision==0.17.2+cu118',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
    )
    print('torch==2.2.2+cu118 + torchvision==0.17.2+cu118 pinned.')
else:
    print('sm_70+ → using default Kaggle torch.')

# ── 4. Uninstall JAX/Flax (conflicts with numpy<2 that torch 2.2.2 needs) ────
print('Uninstalling jax/jaxlib/flax ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax', 'jaxlib', 'flax'],
    check=False,
)
_np_diag = subprocess.run(
    [sys.executable, '-c',
     'import numpy; print("numpy", numpy.__version__, numpy.__file__)'],
    capture_output=True, text=True, timeout=30,
)
print(_np_diag.stdout.strip() or _np_diag.stderr.strip())

# ── 5. NCCL stub (sm_60 only) ─────────────────────────────────────────────────
# torch==2.2.2+cu118 has undefined nccl* symbols; no matching libnccl.so on P100.
if _sm_ver < 70:

    def _find_libtorch_cuda():
        for _pat in [
            '/usr/local/lib/python*/dist-packages/torch/lib/libtorch_cuda.so',
            '/opt/conda/lib/python*/site-packages/torch/lib/libtorch_cuda.so',
        ]:
            _hits = _glob.glob(_pat)
            if _hits:
                return _hits[0]
        raise FileNotFoundError('libtorch_cuda.so not found — torch install failed')

    def _nccl_undef_syms(path):
        r = subprocess.run(['objdump', '-T', path],
                           capture_output=True, text=True, timeout=60)
        syms = {}
        for line in r.stdout.splitlines():
            if '*UND*' not in line:
                continue
            m = re.search(r'\(([\w.]+)\)\s+(nccl\w+)', line)
            if m:
                syms[m.group(2)] = m.group(1); continue
            m2 = re.search(r'\*UND\*\s+\S+\s+(NCCL_[\d.]+)\s+(nccl\w+)', line)
            if m2:
                syms[m2.group(2)] = m2.group(1); continue
            m3 = re.search(r'\s+(nccl\w+)\s*$', line)
            if m3:
                syms.setdefault(m3.group(1), None)
        if not syms:
            r2 = subprocess.run(['nm', '-D', path],
                                capture_output=True, text=True, timeout=60)
            for line in r2.stdout.splitlines():
                parts = line.split()
                if len(parts) >= 2 and parts[-2] == 'U' and parts[-1].startswith('nccl'):
                    syms[parts[-1]] = None
        return syms

    _libtorch_cuda = _find_libtorch_cuda()
    print(f'Scanning NCCL symbols in: {_libtorch_cuda}')
    _nccl_syms = _nccl_undef_syms(_libtorch_cuda)
    print(f'Found {len(_nccl_syms)} undefined nccl* symbols')

    _c_lines = ['#include <stddef.h>', '']
    for _sym, _ver in sorted(_nccl_syms.items()):
        _internal = f'_stub_{_sym}'
        _c_lines.append(f'int {_internal}(void) {{ return 0; }}')
        if _ver:
            _c_lines.append(f'__asm__(".symver {_internal},{_sym}@@{_ver}");')
        else:
            _c_lines.append(
                f'extern int {_sym}(void) __attribute__((alias("{_internal}")));'
            )
        _c_lines.append('')

    _stub_c  = '/tmp/_nccl_stub.c'
    _stub_so = '/tmp/_nccl_stub.so'
    with open(_stub_c, 'w') as _f:
        _f.write('\n'.join(_c_lines))

    _gcc = subprocess.run(
        ['gcc', '-shared', '-fPIC', '-nostartfiles', '-o', _stub_so, _stub_c],
        capture_output=True, text=True, timeout=30,
    )
    if _gcc.returncode != 0:
        print(f'GCC stderr:\n{_gcc.stderr.strip()}')
        raise RuntimeError('NCCL stub compilation failed')

    ctypes.CDLL(_stub_so, mode=ctypes.RTLD_GLOBAL)
    print(f'NCCL stub loaded (RTLD_GLOBAL): {_stub_so}')

    import torch as _torch_smoke
    print(f'torch {_torch_smoke.__version__} imported OK')
    _cuda_ok = _torch_smoke.cuda.is_available()
    _dev = _torch_smoke.cuda.get_device_name(0) if _cuda_ok else 'N/A'
    print(f'CUDA: available={_cuda_ok}, device={_dev}')
    del _torch_smoke

# ── 6. Set headless rendering backend BEFORE dm_control import ────────────────
os.environ.setdefault('MUJOCO_GL', 'egl')
print(f'MUJOCO_GL={os.environ["MUJOCO_GL"]}')
print('Cell 1 complete.')

In [ ]:
# ── Cell 2: Verify transformers version ───────────────────────────────────────
import transformers
REQUIRED = "4.40.1"
if transformers.__version__ != REQUIRED:
    raise RuntimeError(
        f"transformers {transformers.__version__} installed but need exactly {REQUIRED}.\n"
        "Fix: Runtime -> Disconnect and delete runtime, then re-run Cell 1 first."
    )
print(f"transformers {REQUIRED} confirmed.")

In [ ]:
# ── Cell 3: Clone repo and install vlam_compress ──────────────────────────────
import os, sys, subprocess

IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB  = False
try:
    import google.colab; IS_COLAB = True
except ImportError:
    pass

REPO_DIR = "/kaggle/working/vlam" if IS_KAGGLE else "/content/vlam"

_gh_token = ""
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _gh_token = UserSecretsClient().get_secret("GH_TOKEN")
        print("GH_TOKEN loaded from Kaggle Secrets.")
    except Exception as _e:
        print(f"GH_TOKEN not found in Kaggle Secrets ({_e}).")
elif IS_COLAB:
    try:
        from google.colab import userdata
        _gh_token = userdata.get("GH_TOKEN")
        print("GH_TOKEN loaded from Colab Secrets.")
    except Exception as _e:
        print(f"GH_TOKEN not found in Colab Secrets ({_e}).")
else:
    _gh_token = os.environ.get("GH_TOKEN", "")

if _gh_token:
    REPO_URL = f"https://{_gh_token}@github.com/quantumadventurer11/vw-quantum-vlam-challenge"
else:
    print("No GH_TOKEN — using unauthenticated URL (public repo).")
    REPO_URL = "https://github.com/quantumadventurer11/vw-quantum-vlam-challenge"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
_src_path = os.path.join(REPO_DIR, "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)

print(f"Working directory : {os.getcwd()}")
print(f"Environment       : {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local'}")

In [ ]:
# ── Cell 4: HuggingFace authentication ────────────────────────────────────────
from huggingface_hub import login

_hf_token = ""
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle Secrets.")
    except Exception as _e:
        print(f"HF_TOKEN not found in Kaggle Secrets ({_e}) — proceeding without auth.")
elif IS_COLAB:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded from Colab Secrets.")
    except Exception as _e:
        print(f"HF_TOKEN not found in Colab Secrets ({_e}).")
else:
    import os as _os; _hf_token = _os.environ.get("HF_TOKEN", "")

if _hf_token:
    login(token=_hf_token, add_to_git_credential=False)
else:
    print("No HF_TOKEN — model will attempt unauthenticated download.")

In [ ]:
# ── Cell 5: Locate Phase 2 checkpoints ────────────────────────────────────────
# Priority order:
#   1. Kaggle Dataset (benjaminbrumm/vlam-phase2-checkpoints) — dataset_sources
#   2. Kaggle kernel_sources fallback (benjaminbrumm/vw-phase2-compression)
#   3. Google Drive (Colab)
#   4. Local ./checkpoints (dev)
from pathlib import Path
import os

if os.path.isdir('/kaggle/input/vlam-phase2-checkpoints/compressed_chi64'):
    CHECKPOINTS_BASE = Path('/kaggle/input/vlam-phase2-checkpoints')
    print(f'Kaggle dataset: {CHECKPOINTS_BASE}')
elif os.path.isdir('/kaggle/input/vw-phase2-compression/checkpoints'):
    CHECKPOINTS_BASE = Path('/kaggle/input/vw-phase2-compression/checkpoints')
    print(f'Kaggle kernel-source: {CHECKPOINTS_BASE}')
elif os.path.isdir('/kaggle/input/vw-phase2-compression'):
    CHECKPOINTS_BASE = Path('/kaggle/input/vw-phase2-compression')
    print(f'Kaggle kernel-source (root): {CHECKPOINTS_BASE}')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINTS_BASE = Path('/content/drive/MyDrive/vlam_checkpoints')
        print('Google Drive mounted.')
    except Exception as _e:
        CHECKPOINTS_BASE = Path('checkpoints')
        print(f'Local fallback: {CHECKPOINTS_BASE}  ({_e})')

cores_path = CHECKPOINTS_BASE / 'compressed_chi64' / 'cores.pt'
if cores_path.exists():
    print(f'Found chi=64 cores: {cores_path}  ({cores_path.stat().st_size / 1024**2:.0f} MB)')
else:
    print(f'WARNING: {cores_path} not found.')
    print('  Ensure benjaminbrumm/vlam-phase2-checkpoints is listed in dataset_sources.')


In [ ]:
# ── Cell 6: Imports ────────────────────────────────────────────────────────────
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
import yaml
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

from vlam_compress.mujoco_bridge import (
    make_env, run_episode, action_to_joint_command,
    encode_video, make_overlay_fn, make_side_by_side,
    load_compressed_model, restore_patches,
    RENDER_FPS, RENDER_WIDTH, RENDER_HEIGHT,
)

print(f"PyTorch : {torch.__version__}")
print("All imports OK.")

In [ ]:
# ── Cell 7: Verify GPU + dm_control headless rendering ────────────────────────
import os
assert torch.cuda.is_available(), "No GPU — change runtime to GPU and restart."
props = torch.cuda.get_device_properties(0)
print(f"GPU  : {props.name}  ({props.total_memory / 1024**3:.1f} GB VRAM)")
print(f"CUDA : {torch.version.cuda}")

# Test dm_control rendering (will raise if EGL/OSMesa not available)
print("\nTesting dm_control headless render ...")
_env   = make_env(random_state=0)
_env.reset()
_frame = _env.physics.render(height=64, width=64, camera_id=1)
assert _frame.shape == (64, 64, 3), f"Unexpected frame shape: {_frame.shape}"
print(f"Render test OK — frame shape {_frame.shape}, dtype={_frame.dtype}")
del _env, _frame

In [ ]:
# ── Cell 8: Configuration ─────────────────────────────────────────────────────
import yaml
from pathlib import Path

with open("configs/seeds.yaml") as f:
    cfg = yaml.safe_load(f)

CHI             = 64                    # primary target checkpoint
MODEL_ID        = "openvla/openvla-7b"
HF_DATASET      = "lerobot/toto"        # public; bridge_dataset requires auth
UNNORM_KEY      = "toto"               # action un-normalisation key for lerobot/toto
N_STEPS         = 200                  # physics + render cycles (≈ 8 s at 25 fps)
EPISODE_SEED    = cfg["seeds"][0]      # 42
PREDICT_EVERY_N = 10                   # call model every N steps; hold action between
                                       # calls (200 steps / 10 = 20 GPU calls ≈ 40 s)
RESULTS_DIR     = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"chi             : {CHI}")
print(f"dataset         : {HF_DATASET}  (UNNORM_KEY={UNNORM_KEY!r})")
print(f"n_steps         : {N_STEPS}  ({N_STEPS / 25:.0f} s at 25 fps)")
print(f"predict_every_n : {PREDICT_EVERY_N}  ({N_STEPS // PREDICT_EVERY_N} GPU calls total)")
print(f"episode seed    : {EPISODE_SEED}")

In [ ]:
# ── Cell 9: Synthetic scene (bypasses lerobot/toto CDN) ──────────────────────
# lerobot/toto parquet files are served via HuggingFace Xet CDN, which returns
# 403 in Kaggle environments (SignatureError: invalid key pair id).
# The MuJoCo demo only needs the model to produce 7-DoF actions, so we use a
# synthetic 224×224 image and a fixed language prompt.
# Action-prediction accuracy on real data is covered by Phase 3.
import numpy as np
from PIL import Image

# Checkerboard tabletop scene — non-trivial visual input for OpenVLA
_img_arr = np.zeros((224, 224, 3), dtype=np.uint8)
for _i in range(0, 224, 32):
    for _j in range(0, 224, 32):
        if (_i // 32 + _j // 32) % 2 == 0:
            _img_arr[_i:_i+32, _j:_j+32] = [180, 160, 140]  # light beige
        else:
            _img_arr[_i:_i+32, _j:_j+32] = [100, 80, 60]    # dark brown

FIXED_SAMPLE = {
    "image":     Image.fromarray(_img_arr),
    "language":  "pick up the block and place it on the shelf",
    "action_gt": np.zeros(7, dtype=np.float32),
}

print(f"Synthetic scene : {FIXED_SAMPLE['image'].size}  mode={FIXED_SAMPLE['image'].mode}")
print(f"Language prompt : '{FIXED_SAMPLE['language']}'")
print("lerobot/toto skipped (Xet CDN 403) — real eval in Phase 3.")
get_sample_fn = lambda: FIXED_SAMPLE

In [ ]:
# ── Cell 10: Load compressed model (chi=64) ────────────────────────────────────
# load_compressed_model loads INT8 base + patches layers with chi=64 cores.
model, processor, saved_modules = load_compressed_model(
    chi=CHI,
    checkpoints_base=CHECKPOINTS_BASE,
    model_id=MODEL_ID,
)

n_params = sum(p.numel() for p in model.parameters())
peak_mem = torch.cuda.max_memory_allocated() / 1024**2
print(f"Total params : {n_params / 1e9:.3f}B")
print(f"Peak GPU mem : {peak_mem:.0f} MiB")

In [ ]:
# ── Cell 11: Define predict_fn (GPU inference wrapper) ────────────────────────
import numpy as np
from PIL import Image

def predict_fn(image: Image.Image, language: str) -> np.ndarray:
    inputs = processor(language, image).to("cuda:0")
    with torch.no_grad():
        action = model.predict_action(
            **inputs, unnorm_key=UNNORM_KEY, do_sample=False
        )
    return np.array(action, dtype=np.float32)

# Warm-up
print("Warm-up inference pass ...")
_ = predict_fn(FIXED_SAMPLE["image"], FIXED_SAMPLE["language"])
print("OK")

In [ ]:
# ── Cell 12: Run compressed episode ───────────────────────────────────────────
import time
import numpy as np

print(f"Running chi={CHI} compressed episode ({N_STEPS} steps, predict_every_n={PREDICT_EVERY_N}) ...")
t0 = time.perf_counter()

result_compressed = run_episode(
    predict_fn=predict_fn,
    get_sample_fn=get_sample_fn,
    n_steps=N_STEPS,
    random_state=EPISODE_SEED,
    render_width=RENDER_WIDTH,
    render_height=RENDER_HEIGHT,
    predict_every_n=PREDICT_EVERY_N,
)

elapsed = time.perf_counter() - t0
print(f"Done — {result_compressed['n_frames']} frames in {elapsed:.0f} s")
print(f"Language: '{result_compressed['language'][:60]}'")

arm_arr = np.array(result_compressed["arm_trajectory"])
print(f"Arm range — shoulder1: [{arm_arr[:,0].min():.2f}, {arm_arr[:,0].max():.2f}]")
print(f"            shoulder2: [{arm_arr[:,1].min():.2f}, {arm_arr[:,1].max():.2f}]")
print(f"            elbow    : [{arm_arr[:,2].min():.2f}, {arm_arr[:,2].max():.2f}]")

In [ ]:
# ── Cell 13: Run FP16 baseline episode (same scene, weights restored) ──────────
# "Baseline" here means the full FP16 model without TN patches — not INT8, since
# bitsandbytes INT8 is unavailable on P100 (sm_60).  Same FP16 load as Phase 1.
print("Restoring original FP16 weights for baseline episode ...")
restore_patches(saved_modules)

print(f"Running FP16 baseline episode ({N_STEPS} steps, predict_every_n={PREDICT_EVERY_N}) ...")
t0 = time.perf_counter()

result_baseline = run_episode(
    predict_fn=predict_fn,      # same fn — now uses restored FP16 weights
    get_sample_fn=get_sample_fn,
    n_steps=N_STEPS,
    random_state=EPISODE_SEED,
    render_width=RENDER_WIDTH,
    render_height=RENDER_HEIGHT,
    predict_every_n=PREDICT_EVERY_N,
)

elapsed = time.perf_counter() - t0
print(f"Done — {result_baseline['n_frames']} frames in {elapsed:.0f} s")

In [ ]:
# ── Cell 14: Load compression ratio from Phase 2 stats ────────────────────────
COMPRESSION_RATIO = None
stats_path = RESULTS_DIR / "compression_sweep_stats.json"
if stats_path.exists():
    with open(stats_path) as f:
        _stats = json.load(f)
    COMPRESSION_RATIO = (
        _stats.get("sweep_stats", {})
              .get(str(CHI), {})
              .get("layer_compression_ratio_mean")
    )
    if COMPRESSION_RATIO:
        print(f"Compression ratio from Phase 2: {COMPRESSION_RATIO:.2f}x")
    else:
        print("compression_sweep_stats.json found but chi key missing.")
else:
    print("compression_sweep_stats.json not found — run Phase 2 first.")
    print("Proceeding without compression ratio overlay.")

In [ ]:
# ── Cell 15: Encode individual MP4s ───────────────────────────────────────────
overlay_c = make_overlay_fn(
    label=f"TN chi={CHI}",
    language=result_compressed["language"],
    arm_traj=result_compressed["arm_trajectory"],
    compression_ratio=COMPRESSION_RATIO,
)
overlay_b = make_overlay_fn(
    label="FP16 baseline",
    language=result_baseline["language"],
    arm_traj=result_baseline["arm_trajectory"],
)

out_c = RESULTS_DIR / f"demo_compressed_chi{CHI}.mp4"
out_b = RESULTS_DIR / "demo_baseline_fp16.mp4"

encode_video(result_compressed["frames"], out_c, fps=RENDER_FPS, overlay_fn=overlay_c)
encode_video(result_baseline["frames"],   out_b, fps=RENDER_FPS, overlay_fn=overlay_b)

print(f"Written: {out_c}  (+ .gif)")
print(f"Written: {out_b}  (+ .gif)")

duration_s = result_compressed["n_frames"] / RENDER_FPS
print(f"\nVideo duration: {duration_s:.1f} s  (requirement: >= 5 s)")
assert duration_s >= 5.0, f"Video too short: {duration_s:.1f} s < 5 s"

In [ ]:
# ── Cell 16: Encode side-by-side comparison ────────────────────────────────────
sbs_frames = make_side_by_side(
    frames_a=result_baseline["frames"],
    frames_b=result_compressed["frames"],
    label_a="FP16 baseline",
    label_b=f"TN chi={CHI}",
)

out_sbs = RESULTS_DIR / f"demo_side_by_side_chi{CHI}.mp4"
encode_video(sbs_frames, out_sbs, fps=RENDER_FPS)
print(f"Written: {out_sbs}  (+ .gif)")
print(f"Frame size: {sbs_frames[0].shape}  ({sbs_frames[0].shape[1]}px wide)")

In [ ]:
# ── Cell 17: Arm trajectory plot ──────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

arm_c = np.array(result_compressed["arm_trajectory"])
arm_b = np.array(result_baseline["arm_trajectory"])
steps = np.arange(len(arm_c))

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
joint_names = ["shoulder1", "shoulder2", "elbow"]
colors_c = ["#3a86ff", "#8338ec", "#ff006e"]
colors_b = ["#adb5bd", "#adb5bd", "#adb5bd"]

for j, (ax, name) in enumerate(zip(axes, joint_names)):
    ax.plot(steps, arm_c[:, j], color=colors_c[j], label=f"TN chi={CHI}", linewidth=1.5)
    ax.plot(steps, arm_b[:, j], color=colors_b[j], label="INT8 baseline",
            linewidth=1.0, linestyle="--", alpha=0.7)
    ax.set_ylabel(f"{name}\n(ctrl)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Step")
fig.suptitle(f"Right arm trajectory — chi={CHI} vs INT8 baseline", fontweight="bold")
plt.tight_layout()

plot_path = RESULTS_DIR / f"arm_trajectory_chi{CHI}.png"
plt.savefig(str(plot_path), dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

In [ ]:
# ── Cell 18: Verification checklist ───────────────────────────────────────────
print("── Verification ──────────────────────────────────────────────────────────")

# 1. Duration >= 5 s
dur = result_compressed["n_frames"] / RENDER_FPS
ok_dur = dur >= 5.0
print(f"[{'OK  ' if ok_dur else 'FAIL'}] Video duration >= 5 s  : {dur:.1f} s")

# 2. Non-trivial arm motion (arm range > 0.05 in at least one joint)
arm_c = np.array(result_compressed["arm_trajectory"])
arm_range = arm_c.max(axis=0) - arm_c.min(axis=0)
ok_motion = arm_range.max() > 0.05
print(f"[{'OK  ' if ok_motion else 'FAIL'}] Non-trivial arm motion : max range={arm_range.max():.3f}")

# 3. Baseline vs compressed arm trajectories differ
arm_b = np.array(result_baseline["arm_trajectory"])
n = min(len(arm_c), len(arm_b))
diff = np.abs(arm_c[:n] - arm_b[:n]).mean()
ok_diff = diff > 1e-4
print(f"[{'OK  ' if ok_diff else 'FAIL'}] Compressed != baseline  : mean |diff|={diff:.5f}")

# 4. Output files exist
out_c   = RESULTS_DIR / f"demo_compressed_chi{CHI}.mp4"
out_sbs = RESULTS_DIR / f"demo_side_by_side_chi{CHI}.mp4"
ok_c    = out_c.exists()
ok_sbs  = out_sbs.exists()
print(f"[{'OK  ' if ok_c else 'FAIL'}] demo_compressed_chi{CHI}.mp4 exists")
print(f"[{'OK  ' if ok_sbs else 'FAIL'}] demo_side_by_side_chi{CHI}.mp4 exists")

print("──────────────────────────────────────────────────────────────────────────")

In [ ]:
# ── Cell 19: Save outputs (Kaggle) / Download (Colab) ─────────────────────────
import os, shutil
from pathlib import Path

to_save = [
    RESULTS_DIR / f"demo_compressed_chi{CHI}.mp4",
    RESULTS_DIR / f"demo_compressed_chi{CHI}.gif",
    RESULTS_DIR / f"demo_baseline_fp16.mp4",
    RESULTS_DIR / f"demo_baseline_fp16.gif",
    RESULTS_DIR / f"demo_side_by_side_chi{CHI}.mp4",
    RESULTS_DIR / f"demo_side_by_side_chi{CHI}.gif",
    RESULTS_DIR / f"arm_trajectory_chi{CHI}.png",
]

if IS_KAGGLE:
    # On Kaggle: copy artifacts to /kaggle/working so they appear in kernel output
    out_dir = Path("/kaggle/working")
    print("Copying artifacts to /kaggle/working ...")
    for p in to_save:
        if p.exists():
            dst = out_dir / p.name
            shutil.copy2(str(p), str(dst))
            print(f"  Saved: {dst.name}  ({dst.stat().st_size / 1024**2:.1f} MB)")
        else:
            print(f"  skip (not found): {p.name}")

    print("\nTo download after the kernel completes:")
    print("  kaggle kernels output benjaminbrumm/vw-phase4-visualization -p kaggle_output_phase4/")
    print("  Then: git add -f results/demo_compressed_chi64.mp4 results/demo_side_by_side_chi64.mp4")
    print(f"         git add results/arm_trajectory_chi{CHI}.png")
    print(f"  git commit -m '[phase4] add MuJoCo visualization chi={CHI}'")

else:
    # Colab: trigger browser downloads
    try:
        from google.colab import files as _colab_files
        for p in to_save:
            if p.exists():
                _colab_files.download(str(p))
                print(f"Downloading {p.name} ({p.stat().st_size / 1024**2:.1f} MB)")
            else:
                print(f"  skip (not found): {p.name}")
    except ImportError:
        print("Not running in Colab. Results are at:")
        for p in to_save:
            print(f"  {'found' if p.exists() else 'MISSING'}: {p.resolve()}")

## Results Summary

After running all cells the following artefacts are produced:

| File | Contents |
|---|---|
| `results/demo_compressed_chi64.mp4` | Humanoid driven by TN chi=64 model (+ .gif) |
| `results/demo_baseline_fp16.mp4` | Humanoid driven by FP16 baseline (+ .gif) |
| `results/demo_side_by_side_chi64.mp4` | Side-by-side comparison (+ .gif) |
| `results/arm_trajectory_chi64.png` | Right arm joint state plot |

> **Note**: Baseline is FP16 (not INT8) because bitsandbytes INT8 is unsupported on
> P100 (sm_60). Consistent with Phase 1 baseline — documented in §3.1 + Appendix C.

### Challenge sections satisfied

- **User requirement**: 3D visualization of humanoid executing a task driven by
  the compressed model's outputs, using MuJoCo ✓
- **§5.2** Functional QI component demonstrated in a downstream task ✓
- **§5.5** Qualitative comparison: compressed vs. FP16-baseline arm trajectory ✓